In [ ]:
import pandas as pd
import numpy as np
import math
import networkx as nx
from collections import defaultdict
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import pairwise_distances, precision_score, recall_score, f1_score, accuracy_score
from joblib import Parallel, delayed
from itertools import product

def minkowski_distance(x, y, p):
    """计算两个样本之间的 Minkowski 距离"""
    return np.sum(np.abs(x - y) ** p) ** (1 / p)

def find_maximal_cliques(adj):
    """使用 NetworkX 查找图中的所有极大团"""
    G = nx.Graph(adj)
    cliques = list(nx.find_cliques(G))
    return cliques

def matching_degree(granule, h_G, test_sample, X_train, delta):
    """计算匹配度"""
    # 计算测试样本与 granule 中每个训练样本的距离
    # 如果距离小于 delta，则累加 h_G[x]
    phi_G_y = sum(h_G[x] for x in granule if minkowski_distance(test_sample, X_train[x], 2) < delta)
    return phi_G_y

def weighted_vote(labels, weights):
    """
    执行加权投票，返回权重总和最高的标签。
    如果存在多种可能的标签，选择权重总和最高的标签。
    """
    if not labels or not weights:
        return None
    label_weights = defaultdict(float)
    for label, weight in zip(labels, weights):
        label_weights[label] += weight
    # 找到权重总和最高的标签
    return max(label_weights, key=label_weights.get)

def compute_h_G(granule, X_train, delta):
    """计算每个 granule 的 h_G"""
    f_G = {}
    for x in granule:
        # 计算 granule 中所有样本与样本 x 的距离
        distances = np.linalg.norm(X_train[granule] - X_train[x], axis=1)
        count = np.sum(distances < delta)
        f_G[x] = count / len(granule)
    total_f_G = sum(f_G.values())
    if total_f_G == 0:
        h_G = {x: 0 for x in f_G}
    else:
        h_G = {x: f_G[x] / total_f_G for x in f_G}
    return h_G

def run_model(beta, alpha, X_train, y_train, X_test, y_test):
    """
    运行模型并返回评价指标。
    
    参数:
    - beta: 用于计算 delta 的百分位数
    - alpha: 用于筛选邻居的阈值
    - 其他参数: 数据和预计算的邻域信息
    """    
    # 计算训练集之间的距离矩阵
    n = X_train.shape[0]
    dis_arr = np.zeros((n, n))
    neighborhoods = []
    granules = []
    neighborhoods_label = []
    
    dis_arr = pairwise_distances(X_train, metric='euclidean')
    distances = dis_arr[np.triu_indices_from(dis_arr, k=1)]
    delta = np.percentile(distances, beta)
    
    def process_training_sample(i):
        # 找到与样本 i 距离小于 delta 的所有样本索引
        neighbors = np.where(dis_arr[i] < delta)[0]
        temp_label = y_train[neighbors]
        count = pd.Series(temp_label).value_counts()
        if not count[count > alpha * len(neighbors)].empty:
            label = count[count > alpha * len(neighbors)].index.tolist()[0]
            return i, neighbors, label
        else:
            return None

    # 使用多线程并行处理
    results = Parallel(n_jobs=5)(delayed(process_training_sample)(i) for i in range(n))

    # 过滤结果
    for res in results:
        if res is not None:
            i, neighbors, label = res
            neighborhoods.append(i)
            granules.append(neighbors)
            neighborhoods_label.append(label)

    # 将 neighborhoods 转换为 NumPy 数组以便于索引
    neighborhoods = np.array(neighborhoods)

    k = len(neighborhoods)
    
    # 计算每个 granule 的 h_G
    h_Gs = Parallel(n_jobs=5, backend="loky")(
        delayed(compute_h_G)(granule, X_train, delta) for granule in granules
    )
    
    # 定义预测测试样本的函数
    def predict_sample(i, test_sample):
        # 计算测试样本与所有选定的训练样本的距离
        distances = np.linalg.norm(X_train[neighborhoods] - test_sample, axis=1)
        mask = distances < delta
        if np.any(mask):
            # 存在满足条件的邻居
            # 计算匹配度
            phi_values = [matching_degree(granule, h_Gs[idx], test_sample, X_train, delta) 
                          for idx, granule in enumerate(granules)]
            # 获取匹配度最高的k个 granule 的索引
            top_k_indices = np.argsort(phi_values)[-k:]
            # 获取这些 granule 对应的标签和对应的 phi_values
            top_k_labels = [neighborhoods_label[idx] for idx in top_k_indices]
            top_k_weights = [phi_values[idx] for idx in top_k_indices]
            # 执行加权投票
            res = weighted_vote(top_k_labels, top_k_weights)
            return (y_test[i], res)
        else:
            # 无法预测
            return (y_test[i], None)
    
    # 并行预测所有测试样本
    test_results = Parallel(n_jobs=5, backend="loky")(
        delayed(predict_sample)(i, t) for i, t in enumerate(X_test)
    )
    
    # 收集预测结果和真实标签
    y_true = []
    y_pred = []
    for true_label, pred_label in test_results:
        y_true.append(true_label)
        y_pred.append(pred_label)
    
    # 过滤无法预测的样本
    y_true_filtered = []
    y_pred_filtered = []
    for true_label, pred_label in zip(y_true, y_pred):
        if pred_label is not None:
            y_true_filtered.append(true_label)
            y_pred_filtered.append(pred_label)

    # 计算评价指标
    if y_pred_filtered:
        coverage = len(y_pred_filtered) / len(y_test)
        accuracy = accuracy_score(y_true_filtered, y_pred_filtered)
        precision = precision_score(y_true_filtered, y_pred_filtered, average='weighted', zero_division=0)
        recall = accuracy * coverage
        f1 = 2 * (precision * recall) / (precision + recall)
    else:
        coverage = 0
        accuracy = 0
        precision = 0
        recall = 0
        f1 = 0
    
    return {
        'beta': beta,
        'alpha': alpha,
        'coverage': coverage,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }

def hyperparameter_tuning(beta_values, alpha_values, X_train, y_train, X_test, y_test):
    """
    对模型进行超参数调优，搜索最佳的 beta 和 alpha 组合。
    
    参数:
    - beta_values: beta 的候选值列表
    - alpha_values: alpha 的候选值列表
    - 其他参数: 数据集
    """
    
    # 生成所有 beta 和 alpha 的组合
    param_combinations = list(product(beta_values, alpha_values))
    
    # 并行执行超参数组合
    results = Parallel(n_jobs=3, backend="loky")(
        delayed(run_model)(beta, alpha, X_train, y_train, X_test, y_test)
        for beta, alpha in param_combinations
    )
    
    # 转换结果为 DataFrame
    results_df = pd.DataFrame(results)
    
    return results_df

def main():
    # 读取并预处理数据
    data = pd.read_csv('brest_s.csv')
    categorical_cols = data.select_dtypes(include=['object', 'category']).columns
    for col in categorical_cols:
        label_encoder = LabelEncoder()
        data[col]=label_encoder.fit_transform(data[col])
    y = data.iloc[:, -1].to_numpy()
    scaler = MinMaxScaler()
    X = scaler.fit_transform(data.iloc[:, :-1])
    
    # 划分训练集和测试集
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
    
    # 定义超参数搜索范围
    beta_values = np.arange(0.1, 5.1, 0.1).tolist()  # [0.1, 0.2, ..., 5.0]
    alpha_values = np.arange(0.5, 1.0, 0.05).tolist()   # [0.5, 0.55, ..., 0.95]
    
    print("开始超参数调优...")
    
    # 进行超参数调优
    results_df = hyperparameter_tuning(beta_values, alpha_values, X_train, y_train, X_test, y_test)
    
    # 输出所有组合的评价指标
    print("\n所有超参数组合的评价指标:")
    print(results_df)
    
    # 找到最佳的超参数组合，基于 F1 分数
    best_result = results_df.loc[results_df['f1_score'].idxmax()]
    print("\n最佳超参数组合:")
    print(best_result)
    
    # 详细展示最佳组合的指标
    print(f"\n最佳组合的详细指标:\nBeta: {best_result['beta']}\nAlpha: {best_result['alpha']}\n"
          f"Accuracy: {best_result['accuracy']:.4f}\nPrecision: {best_result['precision']:.4f}\n"
          f"Recall: {best_result['recall']:.4f}\nF1 Score: {best_result['f1_score']:.4f}\n"
          f"Coverage: {best_result['coverage']:.4f}")

if __name__ == "__main__":
    main()

开始超参数调优...
